# 08 — DiD-Calibrated Scenario Simulation

This notebook uses the causal DiD estimates from Notebook 07 to project
the impact of three policy scenarios on US food waste:
- **Scenario A**: Federal Date Label Standardization
- **Scenario B**: Nationwide Organic Waste Ban (DiD-calibrated)
- **Scenario C**: Combined (A + B with overlap discount)

Outputs: `scenario_comparison.png`, `scenario_results.json`

---
## 1. Setup & Load Inputs

In [1]:
import os, sys
# Ensure we're at project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

In [2]:
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json

SECTOR_COLORS = {
    'Farm': '#7BA17D', 'Foodservice': '#D4836D',
    'Manufacturing': '#5B8BA0', 'Residential': '#D4C97A',
    'Retail': '#6BADA0'
}

os.makedirs('outputs/ml/charts', exist_ok=True)

In [3]:
with open('outputs/causal/did_results.json') as f:
    did = json.load(f)
with open('outputs/ml/forecast_results.json') as f:
    forecast = json.load(f)
with open('outputs/analysis/eda_report.json') as f:
    eda = json.load(f)

state_2024 = pd.read_parquet('outputs/ml/_state_2024_clustered.parquet')
policy = pd.read_csv('data/policy_timeline.csv')
ban_states = set(policy['state'].tolist())
state_df = pd.read_parquet('outputs/ml/_state_df_enriched.parquet')

print(f"Loaded {len(ban_states)} ban states")
print(f"State panel shape: {state_df.shape}")

Loaded 12 ban states
State panel shape: (216930, 38)


---
## 2. DiD Estimates

Extract the TWFE DiD coefficient for landfill rate. This measured causal
effect is used to calibrate the nationwide ban scenario.

In [4]:
beta_landfill_rate = did['twfe_did']['landfill_rate']['beta']
beta_magnitude = abs(beta_landfill_rate)

print(f"DiD beta (landfill_rate): {beta_landfill_rate:.6f}")
print(f"Using magnitude for scenario: {beta_magnitude:.6f}")

bau_2030 = forecast['bau_2030_tons_landfill']
current_2024 = forecast['current_2024_tons_landfill']
print(f"BAU 2030 landfill: {bau_2030/1e6:.1f}M tons")
print(f"Current 2024 landfill: {current_2024/1e6:.1f}M tons")

DiD beta (landfill_rate): 0.020266
Using magnitude for scenario: 0.020266
BAU 2030 landfill: 26.5M tons
Current 2024 landfill: 23.0M tons


---
## 3. Scenario A: Federal Date Label Standardization

Standardizing "Best By" / "Use By" labels reduces consumer confusion,
diverting an estimated 425,000 tons/year from Retail and Residential waste.

In [5]:
date_label_total = 425000  # tons/year

retail_res = state_df[
    (state_df['year'] == 2024) &
    (state_df['sector'].isin(['Retail', 'Residential']))
].groupby('state').agg(
    tons_waste=('tons_waste', 'sum'),
    co2e=('surplus_total_100_year_mtco2e_footprint', 'sum'),
    us_dollars=('us_dollars_surplus', 'sum'),
    meals_wasted=('meals_wasted', 'sum')
).reset_index()

total_retail_res_waste = retail_res['tons_waste'].sum()
retail_res['share'] = retail_res['tons_waste'] / total_retail_res_waste
retail_res['tons_diverted_a'] = retail_res['share'] * date_label_total
retail_res['co2e_per_ton'] = retail_res['co2e'] / retail_res['tons_waste']
retail_res['ghg_saved_a'] = retail_res['tons_diverted_a'] * retail_res['co2e_per_ton']
retail_res['dollars_per_ton'] = retail_res['us_dollars'] / retail_res['tons_waste']
retail_res['dollars_saved_a'] = retail_res['tons_diverted_a'] * retail_res['dollars_per_ton']
retail_res['meals_per_ton'] = retail_res['meals_wasted'] / retail_res['tons_waste']
retail_res['meals_recovered_a'] = retail_res['tons_diverted_a'] * retail_res['meals_per_ton']

scenario_a = {
    'tons_diverted_national': round(date_label_total, 0),
    'ghg_saved_mt_co2e': round(retail_res['ghg_saved_a'].sum(), 0),
    'dollars_saved_billions': round(retail_res['dollars_saved_a'].sum() / 1e9, 3),
    'meals_recovered': round(retail_res['meals_recovered_a'].sum(), 0),
    'top_5_states': retail_res.nlargest(5, 'tons_diverted_a')[['state', 'tons_diverted_a']].apply(
        lambda r: {'state': r['state'], 'tons_diverted': round(r['tons_diverted_a'], 0)}, axis=1
    ).tolist()
}
print(f"Scenario A — Date Labels: {date_label_total:,} tons diverted")
print(f"  GHG saved: {scenario_a['ghg_saved_mt_co2e']:,.0f} MT CO2e")
print(f"  Dollars saved: ${scenario_a['dollars_saved_billions']:.2f}B")

Scenario A — Date Labels: 425,000 tons diverted
  GHG saved: 1,892,889 MT CO2e
  Dollars saved: $2.82B


---
## 4. Scenario B: Nationwide Organic Waste Ban

Apply the DiD-estimated treatment effect to all non-ban states.
The beta coefficient from TWFE measures the causal reduction in landfill
rate attributable to organic waste bans.

In [6]:
non_ban_states = state_2024[state_2024['has_ban'] == 0].copy()
print(f"Non-ban states: {len(non_ban_states)}")

state_all_2024 = state_df[state_df['year'] == 2024].groupby('state').agg(
    tons_waste=('tons_waste', 'sum'),
    tons_landfill=('tons_landfill', 'sum'),
    tons_surplus=('tons_surplus', 'sum'),
    co2e=('surplus_total_100_year_mtco2e_footprint', 'sum'),
    us_dollars=('us_dollars_surplus', 'sum'),
    meals_wasted=('meals_wasted', 'sum')
).reset_index()

non_ban_metrics = state_all_2024[~state_all_2024['state'].isin(ban_states)].copy()

non_ban_metrics['tons_diverted_b'] = non_ban_metrics['tons_waste'] * beta_magnitude
non_ban_metrics['new_landfill_rate'] = (
    non_ban_metrics['tons_landfill'] - non_ban_metrics['tons_diverted_b']
) / non_ban_metrics['tons_waste']
non_ban_metrics['ghg_saved_b'] = non_ban_metrics['tons_diverted_b'] * (
    non_ban_metrics['co2e'] / non_ban_metrics['tons_waste']
)
non_ban_metrics['dollars_saved_b'] = non_ban_metrics['tons_diverted_b'] * (
    non_ban_metrics['us_dollars'] / non_ban_metrics['tons_surplus']
)
non_ban_metrics['meals_recovered_b'] = non_ban_metrics['tons_diverted_b'] * (
    non_ban_metrics['meals_wasted'] / non_ban_metrics['tons_waste']
)

by_state_b = {}
for _, row in non_ban_metrics.iterrows():
    by_state_b[row['state']] = {
        'tons_diverted': round(row['tons_diverted_b'], 0),
        'ghg_saved_mt_co2e': round(row['ghg_saved_b'], 0),
        'dollars_saved': round(row['dollars_saved_b'], 0)
    }

scenario_b = {
    'did_coefficient_used': beta_landfill_rate,
    'beta_magnitude_applied': beta_magnitude,
    'non_ban_states_affected': len(non_ban_metrics),
    'tons_diverted_national': round(non_ban_metrics['tons_diverted_b'].sum(), 0),
    'ghg_saved_mt_co2e': round(non_ban_metrics['ghg_saved_b'].sum(), 0),
    'dollars_saved_billions': round(non_ban_metrics['dollars_saved_b'].sum() / 1e9, 3),
    'meals_recovered': round(non_ban_metrics['meals_recovered_b'].sum(), 0),
    'by_state': by_state_b,
    'top_5_states': non_ban_metrics.nlargest(5, 'tons_diverted_b')[['state', 'tons_diverted_b']].apply(
        lambda r: {'state': r['state'], 'tons_diverted': round(r['tons_diverted_b'], 0)}, axis=1
    ).tolist()
}
print(f"\nScenario B — Nationwide Ban: {scenario_b['tons_diverted_national']:,.0f} tons diverted")
print(f"  GHG saved: {scenario_b['ghg_saved_mt_co2e']:,.0f} MT CO2e")
print(f"  Dollars saved: ${scenario_b['dollars_saved_billions']:.2f}B")

Non-ban states: 38

Scenario B — Nationwide Ban: 779,439 tons diverted
  GHG saved: 3,361,276 MT CO2e
  Dollars saved: $4.68B


---
## 5. Scenario C: Combined

Combine Scenarios A and B with a 15% overlap discount to account for
policies that affect overlapping waste streams (e.g., date labels on
organic products that would also be captured by a ban).

In [7]:
combined_raw = scenario_a['tons_diverted_national'] + scenario_b['tons_diverted_national']
overlap_discount = 0.15
combined_tons = combined_raw * (1 - overlap_discount)

scale_c = combined_tons / combined_raw if combined_raw > 0 else 0.85

scenario_c = {
    'tons_diverted_national': round(combined_tons, 0),
    'ghg_saved_mt_co2e': round(
        (scenario_a['ghg_saved_mt_co2e'] + scenario_b['ghg_saved_mt_co2e']) * scale_c, 0
    ),
    'dollars_saved_billions': round(
        (scenario_a['dollars_saved_billions'] + scenario_b['dollars_saved_billions']) * scale_c, 3
    ),
    'meals_recovered': round(
        (scenario_a['meals_recovered'] + scenario_b['meals_recovered']) * scale_c, 0
    ),
    'overlap_discount': overlap_discount
}
print(f"Scenario C — Combined: {scenario_c['tons_diverted_national']:,.0f} tons diverted")
print(f"  GHG saved: {scenario_c['ghg_saved_mt_co2e']:,.0f} MT CO2e")
print(f"  Dollars saved: ${scenario_c['dollars_saved_billions']:.2f}B")

Scenario C — Combined: 1,023,773 tons diverted
  GHG saved: 4,466,040 MT CO2e
  Dollars saved: $6.37B


---
## 6. Scenario Chart

Bar chart comparing projected 2030 landfill tonnage under each policy
scenario versus business-as-usual.

In [8]:
fig, ax = plt.subplots(figsize=(12, 7))

categories = ['BAU 2030', 'A: Date Labels', 'B: Nationwide Ban', 'C: Combined']
bau_val = bau_2030
scenario_landfill = [
    bau_val,
    bau_val - scenario_a['tons_diverted_national'],
    bau_val - scenario_b['tons_diverted_national'],
    bau_val - scenario_c['tons_diverted_national']
]
tons_diverted = [
    0, scenario_a['tons_diverted_national'],
    scenario_b['tons_diverted_national'], scenario_c['tons_diverted_national']
]
ghg_saved = [
    0, scenario_a['ghg_saved_mt_co2e'],
    scenario_b['ghg_saved_mt_co2e'], scenario_c['ghg_saved_mt_co2e']
]
dollars_saved = [
    0, scenario_a['dollars_saved_billions'],
    scenario_b['dollars_saved_billions'], scenario_c['dollars_saved_billions']
]

colors_bar = ['#B8B8B8', '#5B8BA0', '#D4836D', '#4A6B6F']
bars = ax.bar(categories, [v / 1e6 for v in scenario_landfill],
              color=colors_bar, width=0.6, edgecolor='white', linewidth=1.5)

ax.axhline(y=current_2024 / 1e6, color='orange', linestyle='--',
           linewidth=1.5, alpha=0.7, label=f'2024 Level ({current_2024/1e6:.1f}M)')

for i, (bar, div, ghg, dol) in enumerate(zip(bars, tons_diverted, ghg_saved, dollars_saved)):
    height = bar.get_height()
    if i == 0:
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.3,
                f'{height:.1f}M tons\n(Business as Usual)',
                ha='center', va='bottom', fontsize=9, fontweight='bold', color='#B8B8B8')
    else:
        ax.text(bar.get_x() + bar.get_width() / 2, height + 0.3,
                f'{height:.1f}M tons\n-{div/1e6:.2f}M diverted\n-{ghg/1e6:.1f}M MT CO2e\n${dol:.1f}B saved',
                ha='center', va='bottom', fontsize=8, fontweight='bold', color=colors_bar[i])

ax.set_ylabel('Tons to Landfill (Millions)', fontsize=12)
ax.set_title('Policy Scenario Impact on US Food Waste to Landfill (2030)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(scenario_landfill) / 1e6 * 1.35)
plt.savefig('outputs/ml/charts/scenario_comparison.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print('Saved scenario_comparison.png')

Saved scenario_comparison.png


---
## 7. Save Results

In [9]:
scenario_results = {
    'bau_2030_tons_landfill': bau_2030,
    'current_2024_tons_landfill': current_2024,
    'scenario_A_date_labels': scenario_a,
    'scenario_B_nationwide_ban': scenario_b,
    'scenario_C_combined': scenario_c
}

with open('outputs/ml/scenario_results.json', 'w') as f:
    json.dump(scenario_results, f, indent=2)
print('Saved scenario_results.json')

Saved scenario_results.json
